In [4]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, cohen_kappa_score

mlflow.set_experiment("Model_Selection_Experiment")

def log_model(model, model_name, X_train, X_test, y_train, y_test):
    
    with mlflow.start_run(run_name=model_name):
        
        # Train
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Metrics
        acc = accuracy_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        kappa = cohen_kappa_score(y_test, y_pred)
        
        # Log params
        mlflow.log_param("model", model_name)
        
        # Log metrics
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("kappa", kappa)
        
        # Log model
        mlflow.sklearn.log_model(model, "model")

2026/04/14 22:23:50 INFO mlflow.tracking.fluent: Experiment with name 'Model_Selection_Experiment' does not exist. Creating a new experiment.


In [1]:
import os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Data loading
data_path = r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\interim\fraud_data"
x_train = pd.read_csv(os.path.join(data_path,"X_train.csv"))
x_test  = pd.read_csv(os.path.join(data_path,"X_test.csv"))
y_train = pd.read_csv(os.path.join(data_path,"y_train.csv"))
y_test  = pd.read_csv(os.path.join(data_path,"y_test.csv"))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
models = {
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced'),
    "RandomForest": RandomForestClassifier(class_weight='balanced'),
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight),
    "LightGBM": LGBMClassifier(class_weight='balanced')
}

for name, model in models.items():
    log_model(model, name, x_train, x_test, y_train, y_test)

import numpy as np

def log_with_threshold(model, model_name, threshold):
    
    with mlflow.start_run(run_name=f"{model_name}_thr_{threshold}"):
        
        y_prob = model.predict_proba(x_test)[:,1]
        y_pred = (y_prob > threshold).astype(int)
        
        mlflow.log_param("threshold", threshold)
        mlflow.log_metric("recall", recall_score(y_test, y_pred))
        mlflow.log_metric("precision", precision_score(y_test, y_pred))

2026/04/14 22:24:15 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/04/14 22:24:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

[LightGBM] [Info] Number of positive: 1805, number of negative: 442770
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.080791 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3373
[LightGBM] [Info] Number of data points in the train set: 444575, number of used features: 81
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


2026/04/14 22:26:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 22:26:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
